In [1]:
import sys
from pathlib import Path

current_path = Path.cwd()
PROJECT_ROOT = None

for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT)) # insert(0) garante prioridade absoluta
    print(f"✅ Raiz do projeto adicionada ao sys.path: {PROJECT_ROOT}")
else:
    print("❌ ERRO: Não foi possível encontrar a raiz do projeto.")

✅ Raiz do projeto adicionada ao sys.path: c:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-Rainfall\RAC-semiurban-forest-ML-2026


# Parallel Pipeline

In [15]:
%%writefile chavascal_pipeline.py
import sys
import gc
from pathlib import Path
from typing import Dict, Any, List

# ====================================================================
# 1. PATH INJECTION FOR WORKERS
# Child processes must discover the project root independently.
# ====================================================================
current_path = Path.cwd()
for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

# ====================================================================
# 2. LIBRARY IMPORTS AND CONFIGURATION
# ====================================================================
from rainfall_acoustic_classification.core import load_audio_sample
from rainfall_acoustic_classification.processing import (
    AudioAugmenter, AudioSegmenter, AcousticMetrics
)

augmenter = AudioAugmenter(sample_rate=24000, noise_prob=0.2, cutout_prob=0.3, random_state=42)
segmenter = AudioSegmenter(sample_rate=24000,segment_duration=10.0, overlap=0.5)
metrics_ext = AcousticMetrics(sample_rate=24000, fft_window_size=1024)

def audio_processing_worker(row_dict: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Worker function to execute signal processing and feature extraction 
    for a single audio file.

    This function loads the audio, applies stochastic augmentation (if flagged),
    segments the signal, and extracts acoustic metrics for each segment.

    Parameters
    ----------
    row_dict : dict
        A dictionary representing a single row from the metadata DataFrame. Must contain
        at least the 'file_path', 'split', and 'should_augment' keys.

    Returns
    -------
    list of dict
        A list of dictionaries, where each dictionary contains the extracted features
        and metadata for a specific segment of the original audio file. Returns an
        empty list if the audio loading fails.
    """
    sample = load_audio_sample(row_dict.get('file_path'), sample_rate=24000)
    if not sample: 
        return []
    
    y = sample.audio_data
    aug_log = "raw"
    
    # Apply stochastic augmentation strictly if the flag is True
    if row_dict.get('split') == 'train' and row_dict.get('should_augment', False):
        y, aug_log = augmenter.process(y)
        
    results = []
    chunks = segmenter.process(y)
    
    for idx, (chunk_array, offset) in enumerate(chunks):
        features = metrics_ext.calculate(chunk_array)
        
        segment_data = row_dict.copy()
        segment_data.update({
            'segment_idx': idx, 
            'offset_sec': offset, 
            'aug_params': aug_log
        })
        segment_data.update(features)
        results.append(segment_data)
        
    # =======================================================
    # AGGRESSIVE GARBAGE COLLECTION
    # =======================================================
    del y
    del chunks
    del sample
    gc.collect() 
    
    return results

Overwriting chavascal_pipeline.py


# Processing

In [16]:
import pandas as pd
from rainfall_acoustic_classification.utils import parallel_pipe
from chavascal_pipeline import audio_processing_worker
from rainfall_acoustic_classification.utils import prepare_balanced_training_set

split_file_path = PROJECT_ROOT / "data" / "splits"
metrics_file_path = PROJECT_ROOT / "data" / "processed" / "Chavascal"

## Train

In [17]:
train_file_path = split_file_path / "chavascal_train.csv"

# Load and tag the split
df_train_meta = pd.read_csv(train_file_path)
df_train_meta['split'] = 'train'

# Define the rain classes according to the methodology
RAIN_CLASSES = ['light', 'moderate', 'heavy', 'violent']

# Apply the balancing function
df_train_final = prepare_balanced_training_set(
    df_train=df_train_meta, 
    rain_classes=RAIN_CLASSES, 
    random_state=42
)

print(f"\n Total tasks dispatched to workers: {len(df_train_final)} files.")


📊 Intra-Class Balancing Strategy:
   -> Majority Class: 'moderate' with N_max = 55 samples.
   -> [light] Originals: 44 | Augmented generated: +11 | Final Total: 55
   -> [moderate] Originals: 55 | Already at N_max. No augmentation applied.
   -> [heavy] Originals: 41 | Augmented generated: +14 | Final Total: 55
   -> [violent] Originals: 7 | Augmented generated: +48 | Final Total: 55

 Total tasks dispatched to workers: 367 files.


In [18]:
# Execute the parallel pipeline
df_train_final_features = df_train_final.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extracting Chavascal Features"
)

Extracting Chavascal Features:   0%|          | 0/367 [00:00<?, ?file/s]

In [19]:
train_metrics_file_path = metrics_file_path / "chavascal_train_metrics.csv"
df_train_final_features.to_csv(train_metrics_file_path, index=False)

## Validation

In [20]:
val_file_path = split_file_path / "chavascal_val.csv"

df_val_meta = pd.read_csv(val_file_path)
df_val_meta['split'] = 'val'

df_val_final_features = df_val_meta.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extraindo Features Chavascal"
)

Extraindo Features Chavascal:   0%|          | 0/37 [00:00<?, ?file/s]

In [21]:
val_metrics_file_path = metrics_file_path / "chavascal_val_metrics.csv"
df_val_final_features.to_csv(val_metrics_file_path, index=False)

## Test

In [22]:
test_file_path = split_file_path / "chavascal_test.csv"

df_test_meta = pd.read_csv(test_file_path)
df_test_meta['split'] = 'test'

df_test_final_features = df_test_meta.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extraindo Features Chavascal"
)

Extraindo Features Chavascal:   0%|          | 0/37 [00:00<?, ?file/s]

In [23]:
test_metrics_file_path = metrics_file_path / "chavascal_test_metrics.csv"
df_test_final_features.to_csv(test_metrics_file_path, index=False)

In [30]:
count_train = df_train_final_features['category'].value_counts()
print('TRAIN:')
print(count_train)
print("Total Samples    ", len(df_train_final_features['category']), '\n')

count_val = df_val_final_features['category'].value_counts()
print('VALIDATION:')
print(count_val)
print("Total Samples    ", len(df_val_final_features['category']), '\n')

count_test = df_test_final_features['category'].value_counts()
print('TEST')
print(count_test)
print("Total Samples    ", len(df_test_final_features['category']), '\n')

total = len(df_test_final_features['category']) + len(df_val_final_features['category']) + len(df_train_final_features['category'])

print("PERCENTAGES:")
print("Train         ", round((len(df_train_final_features['category'])/total), 2))
print("Validation    ", round((len(df_val_final_features['category'])/total), 2))
print("Test          ", round((len(df_test_final_features['category'])/total), 2))

TRAIN:
category
no-rain     1617
light        605
violent      605
moderate     605
heavy        605
Name: count, dtype: int64
Total Samples     4037 

VALIDATION:
category
no-rain     209
moderate     77
light        55
heavy        55
violent      11
Name: count, dtype: int64
Total Samples     407 

TEST
category
no-rain     198
moderate     77
light        66
heavy        55
violent      11
Name: count, dtype: int64
Total Samples     407 

PERCENTAGES:
Train          0.83
Validation     0.08
Test           0.08


In [26]:
df_train_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl4_std,wav_detail_lvl3_energy,wav_detail_lvl3_std,wav_detail_lvl2_energy,wav_detail_lvl2_std,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd
1201,SMM11307_20240518_163000_1-0_heavy_afternoon.wav,2024-05-18 16:30:00,afternoon,1.0,12.0,heavy,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.009133,0.000070,0.008359,0.000020,0.004455,1.610814e-06,0.001269,0.000180,0.0,0.0
2810,SMM11307_20240524_155000_0-2_light_afternoon.wav,2024-05-24 15:50:00,afternoon,0.2,2.4,light,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.011911,0.000057,0.007540,0.000005,0.002175,4.469883e-07,0.000669,0.000382,0.0,0.0
1220,SMM11307_20240113_003000_0-4_moderate_night.wav,2024-01-13 00:30:00,overnight,0.4,4.8,moderate,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.032050,0.001609,0.040111,0.000231,0.015208,1.794957e-05,0.004237,0.000951,0.0,0.0
1263,SMM11307_20240519_044000_0-0_no-rain_night.wav,2024-05-19 04:40:00,overnight,0.0,0.0,no-rain,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.011792,0.000077,0.008766,0.000017,0.004159,1.287690e-06,0.001135,0.000084,0.0,0.0
3681,SMM11307_20240525_142000_4-6_violent_afternoon...,2024-05-25 14:20:00,afternoon,4.6,55.2,violent,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.127318,0.011242,0.106028,0.002083,0.045641,4.106505e-04,0.020265,0.035760,0.0,0.0


In [27]:
df_val_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl4_std,wav_detail_lvl3_energy,wav_detail_lvl3_std,wav_detail_lvl2_energy,wav_detail_lvl2_std,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd
295,SMM11307_20240113_070000_2-2_heavy_morning.wav,2024-01-13 07:00:00,morning,2.2,26.4,heavy,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.074880,0.007601,0.087182,0.001095,0.033098,0.000312,0.017655,0.006135,0.0,0.0
376,SMM11307_20240113_055000_0-2_light_night.wav,2024-01-13 05:50:00,overnight,0.2,2.4,light,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.029205,0.001035,0.032173,0.000152,0.012331,0.000016,0.003942,0.001234,0.0,0.0
222,SMM11307_20240122_031500_0-4_moderate_night.wav,2024-01-22 03:15:00,overnight,0.4,4.8,moderate,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.014193,0.000277,0.016651,0.000024,0.004884,0.000001,0.001072,0.000285,0.0,0.0
371,SMM11307_20240521_045000_0-0_no-rain_night.wav,2024-05-21 04:50:00,overnight,0.0,0.0,no-rain,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.001310,0.000001,0.001208,0.000003,0.001614,0.000001,0.001059,0.000007,0.0,0.0
117,SMM11307_20240113_031500_5-6_violent_night.wav,2024-01-13 03:15:00,overnight,5.6,67.2,violent,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.112395,0.015462,0.124345,0.003219,0.056733,0.001014,0.031845,0.023769,0.0,0.0


In [28]:
df_test_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl4_std,wav_detail_lvl3_energy,wav_detail_lvl3_std,wav_detail_lvl2_energy,wav_detail_lvl2_std,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd
389,SMM11307_20240113_082500_1-2_heavy_morning.wav,2024-01-13 08:25:00,morning,1.2,14.4,heavy,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.029854,0.001158,0.034033,0.000093,0.009647,1.982278e-06,0.001408,0.004403,0.0,0.0
354,SMM11307_20240519_182000_0-2_light_night.wav,2024-05-19 18:20:00,night,0.2,2.4,light,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.011657,0.000084,0.009156,0.000009,0.003041,9.910344e-07,0.000996,0.000290,0.0,0.0
377,SMM11307_20240113_080000_0-8_moderate_morning.wav,2024-01-13 08:00:00,morning,0.8,9.6,moderate,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.036192,0.001554,0.039422,0.000199,0.014123,3.509051e-06,0.001873,0.001568,0.0,0.0
53,SMM11307_20240520_124500_0-0_no-rain_afternoon...,2024-05-20 12:45:00,afternoon,0.0,0.0,no-rain,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.001744,0.000008,0.002858,0.000088,0.009401,1.076636e-04,0.010376,0.000040,0.0,0.0
90,SMM11307_20240525_142500_10-0_violent_afternoo...,2024-05-25 14:25:00,afternoon,10.0,120.0,violent,SMM11307,Chavascal,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.169991,0.013923,0.117995,0.003458,0.058806,7.800909e-04,0.027930,0.053047,0.0,0.0
